## Intrinsic Value Analysis Model Training

#### Model Layers

Layer 1:
- calculates Intrinsic value with Benjamin Graham's formula
- takes share price, EPS, P/E, AAA corp bond yield from 2017-2021
- adds the calculated IV dollar value to a dataframe

Layer 2:
- Neural Network built with the MLPClassifier
- accepts features: P/E, Graham's IV, and current share price
- takes labels made from real price movement between 2021 to 2025
- outputs a binary integer value (1=Undervalued, 0=Overvalued)
- behaves as if predicting in 2021 and checking results in 2025

#### Imports

In [1]:
from warnings import filterwarnings

import pandas as pd
import joblib
from sqlalchemy import exc, create_engine

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.neural_network import MLPClassifier

#### Loading data from Database

In [2]:
server  = "SERVER=localhost\\SQLEXPRESS"
db_name = "DATABASE=stock_metrics"
driver  = "DRIVER=ODBC Driver 17 for SQL Server"
params  = f"{driver};{server};{db_name};Trusted_Connection=yes;"
con_str = f"mssql+pyodbc:///?odbc_connect={params}"

# tables being pulled from the local sql database:
# stocks (ticker, price, earnings, pe_ratio)
# growth_rates (ticker, growth_rate, corp_bond_yield)
# historic_valuations (ticker, valuation)
def load_data():
    filterwarnings("ignore", category=exc.SAWarning)
    engine = create_engine(con_str)
    query = """
    SELECT
        s.ticker, s.price, s.earnings, s.pe_ratio,
        g.growth_rate, g.corp_bond_yield, h.valuation
    FROM dbo.stocks s
    JOIN dbo.growth_rates g ON s.ticker = g.ticker
    JOIN dbo.historic_valuations h ON s.ticker = h.ticker
    WHERE s.price > 0
    """
    df = pd.read_sql(query, engine)
    return df


stocks = load_data()
stocks['valuation'] = stocks['valuation'].map({
    'undervalued': 1, 'overvalued': 0
})
print(f"total stocks: {len(stocks)}")
print(f"total undervalued: {stocks['valuation'].value_counts().get(1, 0)}")
print(f"total overvalued: {stocks['valuation'].value_counts().get(0, 0)}")

total stocks: 851
total undervalued: 429
total overvalued: 422


#### Model Layer 1: Calculating Intrinsic Value

In [ ]:
# 8.5 represents the P/E ratio of a no-growth company
# 4.4 is the average yield of AAA corporate bonds
# 2.67 is the current yield of AAA corporate bonds (2021)
def calculate_intrinsic_value(row):
    cagr = row['growth_rate'] * 100
    iv = (row['earnings'] * (8.5 + (2 * cagr)) * 4.4) / 2.67
    return float(round(iv, 2))

stocks['iv'] = stocks.apply(calculate_intrinsic_value, axis=1)
stocks.dropna(subset=['iv'], inplace=True)
display(stocks.head())

,ticker,price,earnings,pe_ratio,growth_rate,corp_bond_yield,valuation,iv
0,A,65.05,2.10,27.45,0.36,2.67,1,278.58
1,AAL,48.60,3.91,9.92,-0.22,2.67,1,-228.74
2,AAP,109.63,6.19,19.54,0.08,2.67,0,249.92
3,AAPL,155.15,9.20,16.86,0.56,2.67,1,1826.91
4,ABBV,108.48,3.29,19.41,0.25,2.67,1,317.17


#### Model Layer 2: Pre-Processing

In [4]:
classifier_features = ['pe_ratio', 'iv', 'price']
x = stocks[classifier_features]
y = stocks['valuation']

x_scaled = StandardScaler().fit_transform(x)
x_train, x_test, y_train, y_test = train_test_split(
    x_scaled, y, test_size=0.3, random_state=100
)

#### Model Layer 2: Neural Network - MLP Classifier Based

In [5]:
nn_model = MLPClassifier(
    hidden_layer_sizes=(32, 16, 8), activation='relu', 
    solver='adam', alpha=0.01, learning_rate_init=0.001, 
    max_iter=5000, random_state=100
)
nn_model.fit(x_train, y_train)
predictions = nn_model.predict(x_test)

print("--- NN Evaluation ---")
print(classification_report(y_test, predictions))

--- NN Evaluation ---
              precision    recall  f1-score   support

           0       0.80      0.67      0.73       117
           1       0.75      0.86      0.80       139

    accuracy                           0.77       256
   macro avg       0.77      0.76      0.76       256
weighted avg       0.77      0.77      0.77       256



#### Exporting Model

In [6]:
model_artifacts = {
    'scaler': StandardScaler().fit(x),
    'model': nn_model
}

joblib.dump(model_artifacts, 'iv_analyzer.joblib')
print("Model & Scaler successfully packaged/saved!")

Model & Scaler successfully packaged/saved!
